# ML-04 — Search Intelligence Data Contract

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/Sujan-lab-cell/flyrank-ml-internship/blob/main/work/notebooks/w03_data_contract.ipynb?flush_cache=true)

This skeleton is yours to fill. Work the sections **in order** — each one has a one-line hint. Simple words, honest numbers.

> Working with an AI assistant? Tell it to read `skills/README.md` first and load the one skill this assignment names on its card.

### Unit of Analysis

One row represents one pseudonymized content item (`content_id`) belonging to one pseudonymized client (`client_id`).

### Time Window

For Week 3 feature engineering and model-training preparation, I will use the mid-panel month **March 2026** (`month=2026-03`).

The warehouse covers daily performance data from January 27, 2025 through June 30, 2026. I will use March 2026 for development so that the sealed final month, June 2026, is not used to develop the outcome logic.

In [29]:
!pip install -q duckdb

In [30]:
!pip install -q huggingface_hub

In [31]:
import os, getpass

HF_TOKEN = os.environ.get('HF_TOKEN')
if not HF_TOKEN:
    try:
        from google.colab import userdata
        HF_TOKEN = userdata.get('HF_TOKEN')
    except Exception:
        pass

HF_TOKEN = HF_TOKEN or getpass.getpass('Paste your Hugging Face READ token (hf_...): ')

if HF_TOKEN and HF_TOKEN.startswith('hf_'):
    print(f"✅ Token loaded successfully (starts with {HF_TOKEN[:5]}...)")
else:
    print("❌ Invalid token! Make sure you pasted a valid token starting with 'hf_'")


✅ Token loaded successfully (starts with hf_dY...)


In [32]:
import duckdb

con = duckdb.connect()

con.execute(
    f"CREATE OR REPLACE SECRET hf "
    f"(TYPE huggingface, TOKEN '{HF_TOKEN}')"
)

REL = 'hf://datasets/FlyRank/internship-warehouse'

TABLES = {
    'dim_clients': f"read_parquet('{REL}/dim_clients.parquet')",
    'dim_content': f"read_parquet('{REL}/dim_content.parquet')",
    'fact_daily_sample': f"read_parquet('{REL}/fact_content_daily_performance_sample.parquet')",
    'fact_query_90d': f"read_parquet('{REL}/fact_content_query_90d.parquet')",
}

for name, src in TABLES.items():
    count = con.sql(
        f'SELECT COUNT(*) FROM {src}'
    ).fetchone()[0]

    print(f'{name:22} {count:>12,} rows')

dim_clients                     104 rows
dim_content                 519,606 rows
fact_daily_sample        11,694,072 rows
fact_query_90d            2,414,248 rows


## 2. Fields — feature / label / context / excluded

### Features

For Lane 2, the initial five candidate features are:

- `content_age_days` — number of days since the content was created.
- `days_since_last_update` — number of days since the content was last updated.
- `search_volume` — estimated search volume for the page's target keyword.
- `impressions_90d` — Google Search Console impressions over the available 90-day window.
- `avg_position` — mean Google Search position over the available window.

These features are intended to provide different signals about content freshness, search demand, search exposure, and search position. Before using them for modeling, their availability at the decision moment and time-window alignment must be verified.

### Label / Proxy

- `is_declining_label`

The starter proxy label is defined as:

`is_declining_label = 1` when `trend_direction == "down"`, otherwise `0`.

This is a proxy for identifying pages showing an observed declining trend. It is not itself a feature.

### Context

The following fields provide context for identifying, grouping, or describing the content:

- `content_id`
- `client_id`
- `content_type`
- `main_intent`

`content_id` and `client_id` are pseudonymous identifiers and will be used for grouping, joining, or client-level splitting rather than as model features.

### Excluded

The following fields are deliberately excluded from the model features:

- `trend_direction`
- `trend_pct`
- `impressions_last_30d`
- `clicks_last_30d`
- `impressions_prev_30d`
- `clicks_prev_30d`

`trend_direction` and `trend_pct` are excluded because they are directly involved in defining the declining label and would create target leakage.

The recent 30-day comparison fields are excluded at this stage because their relationship to the label window must be carefully aligned with the decision moment before they can be considered as valid features.

The five candidate features above will be verified against the March 2026 warehouse slice before being used in the final feature frame.

In [33]:
features = [
    'content_age_days',
    'days_since_last_update',
    'search_volume',
    'impressions_90d',
    'avg_position'
]

target = ['is_declining_label']

excluded = [
    'trend_direction',
    'trend_pct',
    'impressions_last_30d',
    'clicks_last_30d',
    'impressions_prev_30d',
    'clicks_prev_30d'
]

print(f"Features count: {len(features)}")
print(f"Target count: {len(target)}")
print(f"Excluded count: {len(excluded)}")

assert len(features) == 5
assert len(set(features).intersection(excluded)) == 0, \
    "Leakage error: Excluded columns in features!"

Features count: 5
Target count: 1
Excluded count: 6


## 3. Verify it with queries (grain, counts, missing values, windows)

Each claim above gets a query cell here. A contract claim without a query next to it is a guess.

### Empirical Verification Claims

1. **Grain:** Each warehouse row represents one `report_date × client_id × content_id` observation. Duplicate combinations should not exist.

2. **March 2026 slice:** The development slice contains only March 2026 observations, and its row count and date range are verified directly from the warehouse.

3. **Availability:** GA4 availability is measured using the `ga4_data_available` field rather than assuming zero values mean missing data. Availability is checked explicitly because some rows can contain zero-filled GA4 values when data is unavailable.

The March 2026 slice is used for development because it is a mid-panel month. The final June 2026 month is kept sealed for later evaluation.

In [36]:
# March 2026 warehouse slice
DAILY = f"read_parquet('{REL}/fact_content_daily_performance/**/*.parquet', hive_partitioning=true)"

march = f"""
SELECT *
FROM {DAILY}
WHERE report_date >= DATE '2026-03-01'
  AND report_date < DATE '2026-04-01'
"""

In [38]:
# Query 1: verify the warehouse grain

grain_check = con.sql(f"""
    SELECT
        report_date,
        client_hash_id,
        content_hash_id,
        COUNT(*) AS row_count
    FROM ({march}) AS march_data
    GROUP BY report_date, client_hash_id, content_hash_id
    HAVING COUNT(*) > 1
    LIMIT 5
""").df()

print("Duplicate grain combinations:")
print(grain_check)

FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

Duplicate grain combinations:
Empty DataFrame
Columns: [report_date, client_hash_id, content_hash_id, row_count]
Index: []


In [39]:
# Query 2: verify row count and date window

march_window = con.sql(f"""
    SELECT
        COUNT(*) AS row_count,
        MIN(report_date) AS min_date,
        MAX(report_date) AS max_date
    FROM ({march}) AS march_data
""").df()

print(march_window)

FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

   row_count   min_date   max_date
0    9841378 2026-03-01 2026-03-31


In [40]:
# Query 3: verify GA4 availability

availability_check = con.sql(f"""
    SELECT
        ga4_data_available,
        COUNT(*) AS row_count
    FROM ({march}) AS march_data
    GROUP BY ga4_data_available
    ORDER BY ga4_data_available
""").df()

print(availability_check)

FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

   ga4_data_available  row_count
0               False    6408671
1                True     413966
2                <NA>    3018741


## Self-check

Before you submit, confirm each line honestly:

- [x] Every section above is filled — markdown thinking AND the code that backs it
- [x] The notebook runs top to bottom with no errors (Runtime → Run all)
- [x] No client names, URLs, or private queries anywhere
- [x] My claims use careful words: observed, measured, directional, decision-support
- [x] Committed to my repo under `work/notebooks/` — then submit your repo URL on the card. Done.